In [14]:
import pandas as pd
import numpy as np

# === load your csv ===
df = pd.read_csv("../data/res/dynamictest/dynamictest.csv")

# === safety: avoid log(0) ===
eps = 1e-12
df["z_lmt"] = -np.log(df["p_lmt"].clip(lower=eps))
df["z_pe"]  = -np.log(df["p_pe"].clip(lower=eps))
df["z"]     = df["z_lmt"] + df["z_pe"]

# === target (performance) ===
target = "accuracy"

# === compute correlations ===
results = {}

def corr(x, y):
    return x.corr(y)

results["LMT"]       = corr(df["z_lmt"], df[target])
results["PE"]        = corr(df["z_pe"], df[target])
results["LMT+PE"]    = corr(df["z"], df[target])

# if you also want raw versions (optional but useful)
results["LMT_raw"]   = corr(df["lmt_mean_score"], df[target])
results["PE_raw"]    = corr(df["entropy_score"], df[target])

# === print nicely ===
print("\n=== Correlation with F1 ===")
for k, v in results.items():
    print(f"{k:10s}: {v:.4f}")


=== Correlation with F1 ===
LMT       : -0.7806
PE        : -0.4996
LMT+PE    : -0.7076
LMT_raw   : -0.8602
PE_raw    : -0.6943


In [15]:
print(df["p_pe"].describe())
print(df["entropy_score"].describe())
print(df["lmt_mean_score"].describe())
print(df["p_lmt"].describe())

count    2013.000000
mean        0.060706
std         0.118713
min         0.005181
25%         0.005181
50%         0.010363
75%         0.072539
max         0.948187
Name: p_pe, dtype: float64
count    2013.000000
mean        0.258751
std         0.086896
min         0.107958
25%         0.203626
50%         0.239846
75%         0.288403
max         0.706776
Name: entropy_score, dtype: float64
count    2013.000000
mean        8.468199
std         3.676007
min         3.157804
25%         5.987445
50%         7.629857
75%         9.929904
max        35.525000
Name: lmt_mean_score, dtype: float64
count    2013.000000
mean        0.170987
std         0.190280
min         0.005181
25%         0.025907
50%         0.093264
75%         0.259067
max         0.937824
Name: p_lmt, dtype: float64


In [16]:
# === IMPORTANT: ensure correct temporal order ===
df = df.sort_values("block_index")

# === compute G slope ===
df["g_slope"] = df["g_stat"].diff()

# drop first NaN
df = df.dropna(subset=["g_slope"])

target = "accuracy"

def corr(x, y):
    return x.corr(y)

print("\n=== Correlation with F1 ===")
print(f"G        : {corr(df['g_stat'], df[target]):.4f}")
print(f"G_slope  : {corr(df['g_slope'], df[target]):.4f}")
print(f"Z        : {corr(df['z_evidence'], df[target]):.4f}")
print(f"Z-nu     : {corr(df['z_evidence'] - df['nu'], df[target]):.4f}")


=== Correlation with F1 ===
G        : -0.3152
G_slope  : -0.7075
Z        : -0.7075
Z-nu     : -0.7075


mu candidate

In [17]:
import argparse
import os
import json
import numpy as np
import pandas as pd


def robust_mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return float(1.4826 * np.median(np.abs(x - med)))


def trimmed_mean(x, proportion=0.10):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    k = int(n * proportion)
    if 2 * k >= n:
        raise ValueError(f"Trimming proportion {proportion} too large for n={n}")
    return float(np.mean(x[k:n - k]))


def compute_nu_candidates(z_ref):
    z_ref = np.asarray(z_ref, dtype=float)

    med = float(np.median(z_ref))
    mean = float(np.mean(z_ref))
    std = float(np.std(z_ref))
    mad_val = robust_mad(z_ref)

    nu_candidates = {
        "median": med,
        "mean": mean,
        "q75": float(np.quantile(z_ref, 0.75)),
        "q80": float(np.quantile(z_ref, 0.80)),
        "q90": float(np.quantile(z_ref, 0.90)),
        "trimmed_mean_10": trimmed_mean(z_ref, 0.10),
        "trimmed_mean_20": trimmed_mean(z_ref, 0.20),
        "median_plus_half_mad": med + 0.5 * mad_val,
        "median_plus_mad": med + 1.0 * mad_val,
        "mean_plus_half_std": mean + 0.5 * std,
        "mean_plus_std": mean + 1.0 * std,
    }

    summary = {
        "n_ref_blocks": int(len(z_ref)),
        "z_ref_mean": mean,
        "z_ref_median": med,
        "z_ref_std": std,
        "z_ref_mad": mad_val,
        "z_ref_min": float(np.min(z_ref)),
        "z_ref_max": float(np.max(z_ref)),
    }

    return summary, nu_candidates


def main():
    parser = argparse.ArgumentParser(description="Compute nu candidates from reference CSV")
    parser.add_argument("--ref_csv", type=str, required=True, help="Path to reference CSV")
    parser.add_argument("--z_col", type=str, default="z_evidence", help="Column name for reference Z values")
    parser.add_argument("--outdir", type=str, default="./nu_candidates_out", help="Output directory")
    args = parser.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    df_ref = pd.read_csv(args.ref_csv)

    if args.z_col not in df_ref.columns:
        raise ValueError(f"Column '{args.z_col}' not found in reference CSV")

    z_ref = df_ref[args.z_col].dropna().to_numpy(dtype=float)

    if len(z_ref) == 0:
        raise ValueError("No valid reference z values found")

    summary, nu_candidates = compute_nu_candidates(z_ref)

    result = {
        "reference_csv": args.ref_csv,
        "z_column": args.z_col,
        "summary": summary,
        "nu_candidates": nu_candidates,
    }

    print("=== Reference summary ===")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k:20s}: {v:.6f}")
        else:
            print(f"{k:20s}: {v}")

    print("\n=== Nu candidates ===")
    for k, v in nu_candidates.items():
        print(f"{k:24s}: {v:.6f}")

    # save csv
    out_csv = os.path.join(args.outdir, "nu_candidates.csv")
    rows = []
    for k, v in nu_candidates.items():
        rows.append({"nu_name": k, "nu_value": v})
    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)

    # save json
    out_json = os.path.join(args.outdir, "nu_candidates.json")
    with open(out_json, "w") as f:
        json.dump(result, f, indent=4)

    print(f"\n[SAVE] CSV saved to {out_csv}")
    print(f"[SAVE] JSON saved to {out_json}")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --ref_csv REF_CSV [--z_col Z_COL]
                             [--outdir OUTDIR]
ipykernel_launcher.py: error: the following arguments are required: --ref_csv


SystemExit: 2

/home/uceehow/.conda/envs/aide/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
